# Keyframes -> SigLIP 2 Giant Embeddings (Kaggle 2x GPU 16GB)

Pipeline nhúng keyframe bằng SigLIP 2 Giant; đầu ra là embedding mới 1536 chiều (KHÔNG trộn với CLIP ViT-B/32 512 chiều):

| Thuoc tinh | Gia tri |
| :--- | :--- |
| Tên file | `{video_id}.npy` (ví dụ `L21_V001.npy`) |
| Kích thước | `[N_keyframes, 1536]` |
| Kiểu dữ liệu | `float16` |
| Chuẩn hóa | L2-normalized (norm = 1.0) |
| Model | `google/siglip2-giant-opt-patch16-384` (SigLIP 2 Giant) |
| Thứ tự dòng | sắp xếp tăng dần theo số trong tên file -> **dòng i của `.npy` = keyframe n = i+1** |

Dataset đang dùng: [`fatle542/aic-dataset`](https://www.kaggle.com/datasets/fatle542/aic-dataset) (~108 GB), đường dẫn sau khi Add Input:

```
/kaggle/input/datasets/fatle542/aic-dataset/Keyframes_L21/keyframes/L21_V001/001.jpg
                                                          /002.jpg ...
```
Tên file là **số thứ tự keyframe `n`** (001, 002, ...) - đúng cột `n` của `map-keyframes/*.csv`,
nên dòng `i` của `.npy` tương ứng với `n = i + 1`, khớp với cách index của `clip-features-32` gốc.

### Cách chạy
1. **Cell CONFIG**: đặt `BATCH_ID = 1` (sau đó là 2, 3, 4 ở các lần chạy sau) - hoặc tự nhập `FOLDERS_OVERRIDE`.
2. Chạy toàn bộ notebook. Hai GPU được chia việc **cân bằng theo số ảnh** (greedy bin-packing), mỗi GPU một process riêng.
3. Cell cuối sẽ nén kết quả -> tải về / lưu thành Kaggle Dataset; ghép cả 4 đợt là đủ toàn bộ keyframe.

> Notebook có thể **tiếp tục (resume)**: video nào đã có `.npy` hợp lệ sẽ được bỏ qua.

## 1. CONFIG - chọn thư mục muốn nhúng

In [ ]:
# =====================================================================
#  CONFIG
# =====================================================================

# --- Model ------------------------------------------------------------
# Cấu hình đã chọn: SigLIP 2 Giant - vision encoder mạnh nhất trong các checkpoint
# SigLIP 2 của notebook. Đầu ra 1536 chiều, L2-normalized, phù hợp cho retrieval hình ảnh
# mới. KHÔNG dùng đầu ra này thay thế trực tiếp cho index CLIP ViT-B/32 512 chiều cũ.
#
#   MODEL_ID                                     dim   img/s/T4  chat luong
#   openai/clip-vit-base-patch32                  512    ~700    (baseline BTC)
#   google/siglip2-base-patch16-384               768    ~180    tốt hơn rõ rệt
#   google/siglip2-so400m-patch14-384            1152     ~35    nhẹ hơn
#   google/siglip2-giant-opt-patch16-384         1536     ~14    đang sử dụng
MODEL_ID   = "google/siglip2-giant-opt-patch16-384"
IMAGE_SIZE = 384          # 224 cho CLIP ViT-B/32, 384 cho các bản -384
EMBED_DTYPE = "fp16"      # "fp16" | "fp32". SigLIP được huấn luyện bằng bf16; T4 không có bf16
                          # nên dùng fp16 + chuẩn hóa ở fp32. Đổi sang fp32 nếu thấy NaN.

# `siglip` -> resize THẲNG về (S,S), mean=std=0.5, KHÔNG center-crop.
# `clip`   -> resize cạnh ngắn + center-crop, mean/std của OpenAI CLIP.
MODEL_FAMILY = "siglip" if "siglip" in MODEL_ID.lower() else "clip"

# --- Chia đợt chạy ----------------------------------------------------
# Trước đây BATCHES là danh sách HARD-CODE gồm 13 folder = 160.823 ảnh. Nhưng
# map-keyframes có 177.321 keyframe (L26 một mình có 79.590, trong khi
# L26_a..d chỉ cộng được 63.092) -> THIẾU 16.498 ảnh của một folder L26_*
# không được liệt kê. Những keyframe đó sẽ KHÔNG CÓ vector -> bị bỏ trống
# 9,3% corpus hình ảnh mà không báo lỗi.
# -> Bỏ hard-code. Cell 4 sẽ quét /kaggle/input rồi chia đợt cân bằng.
NUM_BATCHES = 4
BATCH_ID    = 1           # 1..NUM_BATCHES, đổi sau mỗi lần chạy

# --- Muốn tự nhập folder thì điền list vào đây (bỏ qua BATCH_ID) ------
#     ví dụ: FOLDERS_OVERRIDE = ["Keyframes_L21", "Keyframes_L30"]
FOLDERS_OVERRIDE = None

# --- Đường dẫn -------------------------------------------------------
INPUT_ROOTS = ["/kaggle/input/datasets/fatle542/aic-dataset"]
                            # Dataset Kaggle của bạn; bên trong phải có Keyframes_L21, ...
_tag = MODEL_ID.split("/")[-1]
OUTPUT_DIR   = f"/kaggle/working/{_tag}"              # nơi ghi {video_id}.npy
FRAMEIDX_DIR = f"/kaggle/working/{_tag}-frameidx"     # {video_id}.json

# --- Hiệu năng -------------------------------------------------------
NUM_GPUS      = 2       # tự hạ xuống nếu máy có ít GPU hơn
# Giant @384 trên T4 16 GB: batch 8 ưu tiên ổn định, tránh CUDA out of memory.
# BATCH_SIZE là batch trên MỖI GPU (2 GPU -> 2 worker, mỗi worker batch 8).
BATCH_SIZE    = 8
NUM_WORKERS   = 2       # mỗi process; Kaggle có 4 vCPU -> 2 process x 2 workers
IMAGE_EXTS    = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
SKIP_EXISTING = True    # resume: bỏ qua video đã có .npy đúng kích thước
ZIP_RESULT    = True

import os
from pathlib import Path
if INPUT_ROOTS:
    for _root_text in INPUT_ROOTS:
        _root = Path(_root_text)
        if not _root.is_dir():
            raise FileNotFoundError(f"Không tìm thấy input root: {_root}")
        _keyframe_dirs = sorted(p.name for p in _root.glob("Keyframes_*") if p.is_dir())
        print(f"Đã kiểm tra input root: {_root}")
        print(f"Các thư mục Keyframes_*: {_keyframe_dirs}")
        if not _keyframe_dirs:
            raise FileNotFoundError(
                f"Input root không chứa thư mục Keyframes_*: {_root}")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(FRAMEIDX_DIR, exist_ok=True)
print(f"MODEL_ID   = {MODEL_ID}  ({MODEL_FAMILY}, {IMAGE_SIZE}px, {EMBED_DTYPE})")
print(f"OUTPUT_DIR = {OUTPUT_DIR}")
print(f"BATCH      = {BATCH_ID}/{NUM_BATCHES}" if not FOLDERS_OVERRIDE
      else f"FOLDERS_OVERRIDE = {FOLDERS_OVERRIDE}")

## 2. Kiểm tra môi trường & GPU

In [ ]:
import subprocess, sys, importlib

def _pip(*args):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *args], check=False)

# Môi trường Kaggle thường đã có torch/torchvision/transformers; chỉ cài khi thiếu.
for mod, pkg in [("torch", "torch"), ("torchvision", "torchvision"),
                 ("transformers", "transformers"), ("PIL", "pillow")]:
    try:
        importlib.import_module(mod)
    except ImportError:
        print(f"installing {pkg} ...")
        _pip(pkg)

import torch, torchvision, transformers
print("torch       :", torch.__version__)
print("torchvision :", torchvision.__version__)
print("transformers:", transformers.__version__)
# SigLIP2 (Siglip2Model) cần transformers >= 4.49. Nếu môi trường Kaggle cũ báo
# "Unrecognized model" -> nâng lên trước khi chạy tiếp.
if MODEL_FAMILY == "siglip":
    from packaging.version import Version
    if Version(transformers.__version__.split("+")[0]) < Version("4.49"):
        print(f"transformers {transformers.__version__} < 4.49 -> nâng cấp cho SigLIP2 ...")
        _pip("-U", "transformers")
        importlib.reload(transformers)
        print("transformers ->", transformers.__version__)

print("CUDA        :", torch.cuda.is_available(), "| n_gpu =", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {p.name}  {p.total_memory / 1024**3:.1f} GB")

NUM_GPUS = min(NUM_GPUS, max(1, torch.cuda.device_count()))
print("-> dùng", NUM_GPUS, "GPU")

In [ ]:
# Chỉ đọc config ở process chính; KHÔNG nạp cả Giant vào RAM một lần nữa.
# Mỗi worker bên dưới sẽ nạp model trên một GPU riêng.
# Nếu notebook tắt Internet: Add Input model rồi đặt MODEL_ID = "/kaggle/input/<...>".
from transformers import AutoConfig
_cfg = AutoConfig.from_pretrained(MODEL_ID)
if MODEL_FAMILY == "siglip":
    EMBED_DIM = int(_cfg.vision_config.hidden_size)   # giant=1536, so400m=1152
    print("siglip OK | vision hidden =", EMBED_DIM,
          "| text hidden =", _cfg.text_config.hidden_size)
    assert _cfg.vision_config.image_size == IMAGE_SIZE, (
        f"checkpoint là {_cfg.vision_config.image_size}px nhưng IMAGE_SIZE={IMAGE_SIZE}")
else:
    EMBED_DIM = int(_cfg.projection_dim)
    print("clip OK | projection_dim =", EMBED_DIM)

print(f"-> EMBED_DIM = {EMBED_DIM} (mỗi .npy sẽ là [N, {EMBED_DIM}] float16)")

## 3. Quét dữ liệu - tìm video & keyframe

Notebook chấp nhận **cả hai** biến thể cấu trúc:

```
A) fatle542/aic-dataset (co lop "keyframes/" o giua):
   /kaggle/input/aic-dataset/Keyframes_L21/keyframes/L21_V001/001.jpg, 002.jpg ...

B) output của transnetv2_dake_keyframes.py (không có lớp giữa):
   <INPUT_ROOT>/Keyframes_L21/L21_V001/0.jpg, 90.jpg, 261.jpg ...
```

Cả hai đều được sắp xếp **theo số** trong tên file nên thứ tự dòng luôn đúng.
Cell này còn cảnh báo nếu cách đánh số của một video không liên tục `1..N`
(ví dụ thiếu `007.jpg`) - lúc đó dòng i không còn tương ứng với `n = i+1`, cần đối chiếu
`siglip2-giant-opt-patch16-384-frameidx/{video_id}.json` để ánh xạ lại.

In [ ]:
from pathlib import Path
import json, re

def find_input_roots():
    """Tìm folder cha chứa các Keyframes_*.

    KHÔNG dùng rglob(): dataset ~108 GB / 160k+ file, rglob sẽ quét cả cây thư mục
    (và còn đi xuyên vào trong từng Keyframes_*) -> mất rất nhiều phút trên mount
    của Kaggle. Chỉ glob 3 mức đầu là đủ cho các cách đặt dataset thường gặp.
    """
    if INPUT_ROOTS:
        return [Path(p) for p in INPUT_ROOTS]
    base = Path("/kaggle/input")
    if not base.exists():
        return []
    roots = set()
    for pat in ("Keyframes_*", "*/Keyframes_*", "*/*/Keyframes_*",
                "*/*/*/Keyframes_*", "*/*/*/*/Keyframes_*"):
        for d in base.glob(pat):
            if d.is_dir():
                roots.add(d.parent)
    return sorted(roots)

ROOTS = find_input_roots()
print("Các input root được phát hiện:")
for r in ROOTS:
    print("  ", r)
if not ROOTS:
    raise SystemExit("Không tìm thấy folder Keyframes_* nào trong /kaggle/input. "
                     "Hãy Add Input dataset hoặc đặt INPUT_ROOTS thủ công.")

# ---- QUÉT TẤT CẢ folder Keyframes_* (không dùng danh sách hard-code) --------
ALL_FOLDERS = {}
for r in ROOTS:
    for d in sorted(r.glob("Keyframes_*")):
        if not d.is_dir():
            continue
        inner = d / "keyframes"
        ALL_FOLDERS[d.name] = inner if inner.is_dir() else d
print(f"\n{len(ALL_FOLDERS)} folder Keyframes_*: {sorted(ALL_FOLDERS)}")

def frame_sort_key(p):
    """Sắp xếp theo số trong tên file (`001.jpg` -> 1, `261.jpg` -> 261);
    dùng sắp xếp chuỗi dự phòng nếu tên không có số."""
    m = re.findall(r"\d+", p.stem)
    return (0, int(m[-1]), p.stem) if m else (1, 0, p.stem)

# ---- LIỆT KÊ TOÀN BỘ video/ảnh, rồi mới chia đợt -------------------------
# Phải quét hết trước khi chia; nếu không sẽ lặp lại đúng bug cũ: danh sách folder
# hard-code thiếu một folder L26_* -> 16.498 keyframe không bao giờ được nhúng.
ALL_TASKS, noncontig = [], []
print(f"\n{'Folder':<22} {'Video':>6} {'Ảnh':>9}")
print("-" * 40)
for fname, fdir in ALL_FOLDERS.items():
    vids = sorted([d for d in fdir.iterdir() if d.is_dir()])
    n_img_folder = 0
    for vd in vids:
        imgs = sorted([p for p in vd.iterdir()
                       if p.is_file() and p.suffix.lower() in IMAGE_EXTS],
                      key=frame_sort_key)
        if not imgs:
            continue
        n_img_folder += len(imgs)
        # Kiểm tra cách đánh số 1..N liên tục -> dòng i của .npy tương ứng với keyframe n = i+1
        nums = [int(re.findall(r"\d+", p.stem)[-1]) for p in imgs
                if re.findall(r"\d+", p.stem)]
        if len(nums) == len(imgs) and nums != list(range(1, len(imgs) + 1)):
            noncontig.append((vd.name, len(imgs), nums[0], nums[-1]))
        ALL_TASKS.append({"video_id": vd.name, "folder": fname, "dir": str(vd),
                          "images": [str(p) for p in imgs], "n": len(imgs)})
    print(f"{fname:<22} {len(vids):>6} {n_img_folder:>9,}")
print("-" * 40)
TOTAL_ALL = sum(t["n"] for t in ALL_TASKS)
print(f"{'TỔNG':<22} {len(ALL_TASKS):>6} {TOTAL_ALL:>9,}")
if TOTAL_ALL != 177_321:
    print(f"\n[CẢNH BÁO] map-keyframes có 177.321 keyframe nhưng chỉ thấy {TOTAL_ALL:,} ảnh "
          f"(lệch {177_321 - TOTAL_ALL:,}). Keyframe không có ảnh sẽ phải zero-fill khi "
          f"dùng vision.faiss - xem notebook aic26-01b-vision-reindex.")

if noncontig:
    print(f"\nLưu ý: {len(noncontig)} video có cách đánh số KHÔNG liên tục 1..N "
          f"(dòng i != keyframe n=i+1) -> dùng file frameidx JSON để ánh xạ lại. Ví dụ:")
    for v, n, lo, hi in noncontig[:5]:
        print(f"   - {v}: {n} ảnh, số nhỏ nhất={lo}, lớn nhất={hi}")

# ---- CHIA ĐỢT CÂN BẰNG THEO SỐ ẢNH, theo đơn vị folder -----------------
FOLDER_IMAGE_COUNTS = {}
for t in ALL_TASKS:
    FOLDER_IMAGE_COUNTS[t["folder"]] = FOLDER_IMAGE_COUNTS.get(t["folder"], 0) + t["n"]

if FOLDERS_OVERRIDE:
    SELECTED_FOLDERS = list(FOLDERS_OVERRIDE)
else:
    bins = [[] for _ in range(NUM_BATCHES)]
    load = [0] * NUM_BATCHES
    for f in sorted(FOLDER_IMAGE_COUNTS, key=lambda x: -FOLDER_IMAGE_COUNTS[x]):
        i = min(range(NUM_BATCHES), key=lambda k: load[k])
        bins[i].append(f)
        load[i] += FOLDER_IMAGE_COUNTS[f]
    print(f"\n{NUM_BATCHES} đợt cân bằng (lệch lớn nhất - nhỏ nhất: {max(load)-min(load):,} ảnh):")
    for i, b in enumerate(bins, 1):
        mark = " <-- dot nay" if i == BATCH_ID else ""
        print(f"  đợt {i}: {load[i-1]:>7,} ảnh  {b}{mark}")
    assert 1 <= BATCH_ID <= NUM_BATCHES, f"BATCH_ID phải trong 1..{NUM_BATCHES}"
    SELECTED_FOLDERS = bins[BATCH_ID - 1]

VIDEO_TASKS = [t for t in ALL_TASKS if t["folder"] in set(SELECTED_FOLDERS)]
total_imgs = sum(t["n"] for t in VIDEO_TASKS)
print(f"\nĐợt này: {len(VIDEO_TASKS)} video / {total_imgs:,} ảnh"
      f"  ->  ~{total_imgs // max(NUM_GPUS, 1):,} ảnh / GPU")
assert VIDEO_TASKS, "Không tìm được ảnh nào."

In [ ]:
# --- Resume: loại bỏ video đã nhúng xong ------------------------------
import numpy as np

def already_done(task):
    f = Path(OUTPUT_DIR) / f"{task['video_id']}.npy"
    if not f.exists():
        return False
    try:
        arr = np.load(f, mmap_mode="r")
        return arr.shape == (task["n"], EMBED_DIM) and arr.dtype == np.float16
    except Exception:
        return False

if SKIP_EXISTING:
    before = len(VIDEO_TASKS)
    VIDEO_TASKS = [t for t in VIDEO_TASKS if not already_done(t)]
    print(f"Resume: bỏ qua {before - len(VIDEO_TASKS)} / {before} video đã có .npy hợp lệ")

TODO_IMGS = sum(t["n"] for t in VIDEO_TASKS)
print(f"Cần xử lý: {len(VIDEO_TASKS)} video / {TODO_IMGS:,} ảnh")

## 4. Chia việc cân bằng cho 2 GPU (greedy bin-packing theo số ảnh)

In [ ]:
# Sắp video theo số ảnh giảm dần rồi lần lượt đẩy vào shard đang ít ảnh nhất
# -> hai GPU kết thúc gần như cùng lúc, không tách đôi một video.
shards = [[] for _ in range(NUM_GPUS)]
load = [0] * NUM_GPUS
for t in sorted(VIDEO_TASKS, key=lambda x: -x["n"]):
    i = min(range(NUM_GPUS), key=lambda k: load[k])
    shards[i].append(t)
    load[i] += t["n"]

SHARD_FILES = []
for i, sh in enumerate(shards):
    p = Path(f"/kaggle/working/shard_{i}.json")
    p.write_text(json.dumps(sh), encoding="utf-8")
    SHARD_FILES.append(str(p))
    print(f"GPU {i}: {len(sh):>4} video, {load[i]:>7,} ảnh  -> {p.name}")
print(f"Lệch tải giữa các GPU: {max(load) - min(load):,} ảnh")

## 5. Worker script (một process / một GPU)

In [ ]:
%%writefile /kaggle/working/embed_worker.py
# -*- coding: utf-8 -*-
"""Bộ nhúng ảnh keyframe - một process trên một GPU. Hỗ trợ CLIP và SigLIP/SigLIP2.

Đầu ra: {video_id}.npy, [N, DIM], float16, L2-normalized (giống format clip-features-32
của BTC, chỉ khác DIM khi đổi model).
"""
import argparse, json, time
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image, ImageFile
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T

ImageFile.LOAD_TRUNCATED_IMAGES = True   # ảnh JPEG bị cắt đuôi vẫn đọc được
Image.MAX_IMAGE_PIXELS = None

CLIP_MEAN = (0.48145466, 0.4578275, 0.40821073)
CLIP_STD = (0.26862954, 0.26130258, 0.27577711)


def build_transform(family, size):
    """Phải khớp CHÍNH XÁC processor của từng họ model, nếu không embedding sẽ lệch
    mà không hề báo lỗi.

    - CLIP  (CLIPImageProcessor): resize cạnh ngắn bicubic -> center crop -> mean/std CLIP.
    - SigLIP (SiglipImageProcessor): resize THẲNG về (size,size) (squash, KHÔNG crop)
      -> mean=std=0.5. Center-crop ở đây sẽ cắt mất chữ chạy ngang dưới frame
      (chyron) - đúng thứ tín hiệu mạnh nhất của video thời sự.
    """
    # Không dùng T.Lambda(lambda ...) ở đây: lambda không pickle được nên DataLoader
    # với num_workers>0 sẽ lỗi lúc spawn worker. Chuyển sang RGB trong __getitem__.
    if family == "siglip":
        return T.Compose([
            T.Resize((size, size), interpolation=T.InterpolationMode.BICUBIC),
            T.ToTensor(),
            T.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),
        ])
    return T.Compose([
        T.Resize(size, interpolation=T.InterpolationMode.BICUBIC),
        T.CenterCrop(size),
        T.ToTensor(),
        T.Normalize(CLIP_MEAN, CLIP_STD),
    ])


class FlatImageDataset(Dataset):
    """Toàn bộ ảnh của shard trong một dataset phẳng -> DataLoader workers persistent,
    tránh overhead tạo lại loader cho từng video."""

    def __init__(self, paths, family, size):
        self.paths = paths
        self.size = size
        self.tf = build_transform(family, size)

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        try:
            with Image.open(self.paths[i]) as im:
                return self.tf(im.convert("RGB")), i, 1
        except Exception as e:
            print(f"[warn] lỗi đọc ảnh {self.paths[i]}: {e}", flush=True)
            return torch.zeros(3, self.size, self.size), i, 0


def load_model(mid, family, dtype):
    if family == "siglip":
        from transformers import AutoModel
        m = AutoModel.from_pretrained(mid, torch_dtype=dtype)
        dim = int(m.config.vision_config.hidden_size)
    else:
        from transformers import CLIPModel
        try:
            m = CLIPModel.from_pretrained(mid, torch_dtype=dtype, use_safetensors=True)
        except Exception:
            m = CLIPModel.from_pretrained(mid, torch_dtype=dtype)
        dim = int(m.config.projection_dim)
    # Chỉ cần nhánh ảnh -> giải phóng bớt VRAM
    for attr in ("text_model", "text_projection"):
        if hasattr(m, attr):
            try:
                delattr(m, attr)
            except Exception:
                setattr(m, attr, None)
    return m, dim


def extract_image_tensor(output):
    """Chuẩn hóa đầu ra khác nhau giữa các phiên bản Transformers."""
    if isinstance(output, torch.Tensor):
        return output
    # SigLIP2 trên một số bản Transformers trả về BaseModelOutputWithPooling.
    pooled = getattr(output, "pooler_output", None)
    if isinstance(pooled, torch.Tensor):
        return pooled
    image_embeds = getattr(output, "image_embeds", None)
    if isinstance(image_embeds, torch.Tensor):
        return image_embeds
    last_hidden = getattr(output, "last_hidden_state", None)
    if isinstance(last_hidden, torch.Tensor):
        return last_hidden[:, 0]
    if isinstance(output, (tuple, list)) and output and isinstance(output[0], torch.Tensor):
        return output[0]
    raise TypeError(f"Không nhận diện được kiểu đầu ra của model: {type(output).__name__}")


def encode_with_oom_retry(model, px, dev, dtype, min_batch=1):
    """Mã hóa một batch CPU; tự động chia nhỏ nếu VRAM bị đầy."""
    try:
        px_dev = px.to(dev, dtype=dtype, non_blocking=True)
        raw = model.get_image_features(pixel_values=px_dev)
        return extract_image_tensor(raw).float()
    except RuntimeError as e:
        is_oom = dev.type == "cuda" and "out of memory" in str(e).lower()
        if not is_oom or px.shape[0] <= min_batch:
            raise
        old_batch = px.shape[0]
        mid = max(min_batch, old_batch // 2)
        print(f"[oom-backoff] batch {old_batch} -> {mid} + {old_batch - mid}", flush=True)
        torch.cuda.empty_cache()
        left = encode_with_oom_retry(model, px[:mid], dev, dtype, min_batch)
        right = encode_with_oom_retry(model, px[mid:], dev, dtype, min_batch)
        return torch.cat((left, right), dim=0)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--shard-file", required=True)
    ap.add_argument("--out-dir", required=True)
    ap.add_argument("--frameidx-dir", required=True)
    ap.add_argument("--model-id", default="openai/clip-vit-base-patch32")
    ap.add_argument("--family", default="clip", choices=["clip", "siglip"])
    ap.add_argument("--image-size", type=int, default=224)
    ap.add_argument("--dtype", default="fp16", choices=["fp16", "fp32"])
    ap.add_argument("--batch-size", type=int, default=256)
    ap.add_argument("--workers", type=int, default=2)
    ap.add_argument("--tag", default="gpu?")
    args = ap.parse_args()

    tag = args.tag
    tasks = json.loads(Path(args.shard_file).read_text(encoding="utf-8"))
    if not tasks:
        print(f"[{tag}] shard rỗng, thoát.", flush=True)
        return

    paths, offsets = [], []          # offset của từng video trong mảng phẳng
    for t in tasks:
        offsets.append(len(paths))
        paths.extend(t["images"])
    total = len(paths)
    print(f"[{tag}] {len(tasks)} video / {total:,} ảnh", flush=True)

    dev = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    dtype = torch.float32
    if dev.type == "cuda" and args.dtype == "fp16":
        dtype = torch.float16
    model, DIM = load_model(args.model_id, args.family, dtype)
    model = model.to(dev).eval()
    torch.backends.cudnn.benchmark = True
    if dev.type == "cuda":
        free_vram, total_vram = torch.cuda.mem_get_info(dev)
        print(f"[{tag}] VRAM free after model: {free_vram / 1024**3:.1f}/{total_vram / 1024**3:.1f} GB", flush=True)
    print(f"[{tag}] {args.model_id} ({args.family}) on {dev} {dtype} | dim={DIM}", flush=True)

    loader = DataLoader(
        FlatImageDataset(paths, args.family, args.image_size),
        batch_size=args.batch_size,
        shuffle=False,
        num_workers=args.workers,
        pin_memory=(dev.type == "cuda"),
        persistent_workers=args.workers > 0,
        prefetch_factor=4 if args.workers > 0 else None,
        drop_last=False,
    )

    feats = np.zeros((total, DIM), dtype=np.float16)
    ok_mask = np.zeros(total, dtype=bool)
    n_nan = 0
    done, t0, t_log = 0, time.time(), time.time()

    with torch.inference_mode():
        for px, idx, ok in loader:
            emb = encode_with_oom_retry(model, px, dev, dtype)
            # SigLIP được huấn luyện bằng bf16; chạy fp16 trên T4 có thể overflow -> inf/NaN.
            # Bắt tại đây thay vì để vector rác lọt vào FAISS.
            bad = ~torch.isfinite(emb).all(dim=1)
            if bad.any():
                n_nan += int(bad.sum())
                emb[bad] = 0.0
            emb = F.normalize(emb, dim=-1)                # Chuẩn hóa L2 ở fp32 rồi mới hạ xuống fp16
            ix = idx.numpy()
            feats[ix] = emb.cpu().numpy().astype(np.float16)
            m = ok.numpy().astype(bool)
            m[bad.cpu().numpy()] = False
            ok_mask[ix] = m
            done += px.shape[0]
            if time.time() - t_log > 20 or done == total:
                el = time.time() - t0
                ips = done / max(el, 1e-9)
                eta = (total - done) / max(ips, 1e-9)
                print(f"[{tag}] {done:>7,}/{total:,} ({100 * done / total:5.1f}%) "
                      f"| {ips:6.1f} img/s | elapsed {el / 60:5.1f}m | ETA {eta / 60:5.1f}m",
                      flush=True)
                t_log = time.time()

    if n_nan:
        print(f"[{tag}] {n_nan} vector NaN/inf -> ghi 0. Nếu còn nhiều, "
              f"hãy chạy lại với EMBED_DTYPE='fp32'.", flush=True)

    # Ảnh đọc lỗi đã đi qua model dưới dạng tensor 0 -> tạo ra một vector CHUẨN HÓA nhưng vô nghĩa,
    # có thể lọt vào top-k khi retrieval. Ghi về vector 0 để cosine luôn = 0 (không bao giờ khớp).
    n_failed = int((~ok_mask).sum())
    if n_failed:
        feats[~ok_mask] = 0
        print(f"[{tag}] {n_failed} ảnh lỗi -> ghi vector 0 (cosine = 0, không thể khớp)", flush=True)

    out_dir = Path(args.out_dir); out_dir.mkdir(parents=True, exist_ok=True)
    fi_dir = Path(args.frameidx_dir); fi_dir.mkdir(parents=True, exist_ok=True)

    n_bad_total = 0
    for t, off in zip(tasks, offsets):
        n = t["n"]
        bad = int((~ok_mask[off:off + n]).sum())
        n_bad_total += bad
        np.save(out_dir / f"{t['video_id']}.npy", feats[off:off + n])
        (fi_dir / f"{t['video_id']}.json").write_text(json.dumps({
            "video_id": t["video_id"],
            "folder": t["folder"],
            "n": n,
            "model": args.model_id,
            "family": args.family,
            "image_size": args.image_size,
            "dim": DIM,
            "dtype": "float16",
            "normalized": True,
            "failed_images": bad,
            # dòng i của .npy tương ứng với phần tử i của list này (tên file không có đuôi .jpg)
            # -> với aic-dataset là số keyframe n: ["001", "002", ...]
            "frames": [Path(p).stem for p in t["images"]],
        }, ensure_ascii=False), encoding="utf-8")

    el = time.time() - t0
    print(f"[{tag}] HOÀN TẤT {len(tasks)} video / {total:,} ảnh trong {el / 60:.1f} phút "
          f"({total / max(el, 1e-9):.1f} ảnh/s), ảnh lỗi: {n_bad_total}", flush=True)


if __name__ == "__main__":
    main()

## 6. Chạy song song 2 GPU

Mỗi worker là một process riêng với `CUDA_VISIBLE_DEVICES=<i>` nên chỉ dùng đúng một GPU
(không dùng DataParallel -> không có overhead gom/tách tensor, VRAM được tách riêng).

In [ ]:
import subprocess, threading, time, os, sys

procs, threads = [], []

def pump(proc):
    for line in iter(proc.stdout.readline, ""):
        if line:
            sys.stdout.write(line)
            sys.stdout.flush()

t_start = time.time()
for i, shard_file in enumerate(SHARD_FILES):
    env = dict(os.environ)
    env["CUDA_VISIBLE_DEVICES"] = str(i)          # mỗi process dùng 1 GPU -> cuda:0
    env["TOKENIZERS_PARALLELISM"] = "false"
    env["OMP_NUM_THREADS"] = "2"
    cmd = [sys.executable, "/kaggle/working/embed_worker.py",
           "--shard-file", shard_file,
           "--out-dir", OUTPUT_DIR,
           "--frameidx-dir", FRAMEIDX_DIR,
           "--model-id", MODEL_ID,
           "--family", MODEL_FAMILY,
           "--image-size", str(IMAGE_SIZE),
           "--dtype", EMBED_DTYPE,
           "--batch-size", str(BATCH_SIZE),
           "--workers", str(NUM_WORKERS),
           "--tag", f"gpu{i}"]
    p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    procs.append(p)
    th = threading.Thread(target=pump, args=(p,), daemon=True)
    th.start()
    threads.append(th)
    print(f"-> đã khởi chạy worker GPU {i} (pid {p.pid})")

for th in threads:
    th.join()
codes = [p.wait() for p in procs]
print(f"\nMã thoát: {codes} | tổng thời gian: {(time.time() - t_start) / 60:.1f} phút")
assert all(c == 0 for c in codes), "Có worker lỗi - xem log phía trên."

## 7. Kiểm tra output SigLIP 2 Giant

In [ ]:
import numpy as np
from pathlib import Path

files = sorted(Path(OUTPUT_DIR).glob("*.npy"))
print(f"Số file .npy: {len(files)}")

bad, total_rows = [], 0
for f in files:
    a = np.load(f, mmap_mode="r")
    if a.ndim != 2 or a.shape[1] != EMBED_DIM or a.dtype != np.float16:
        bad.append((f.name, a.shape, str(a.dtype)))
    total_rows += a.shape[0]

print(f"Tổng vector (keyframe): {total_rows:,}")
if bad:
    print("File sai định dạng:")
    for b in bad[:20]:
        print("  ", b)
else:
    print(f"OK - mọi file đều có dạng [N, {EMBED_DIM}] float16")

for f in files[:3]:
    a = np.load(f).astype(np.float32)
    n = np.linalg.norm(a, axis=1)
    print(f"  {f.name:<16} shape={a.shape} norm min/max = {n.min():.4f}/{n.max():.4f}")

# Đối chiếu với dataset gốc nếu có attach (glob có giới hạn độ sâu, không rglob cả 108 GB)
ref = next((c for pat in ("clip-features-32", "*/clip-features-32", "*/*/clip-features-32")
            for c in Path("/kaggle/input").glob(pat) if c.is_dir()), None)
if ref:
    r = sorted(ref.glob("*.npy"))
    if r:
        ra = np.load(r[0], mmap_mode="r")
        print(f"\nReference {r[0].name}: shape={ra.shape} dtype={ra.dtype}  <- chỉ để đối chiếu, không trộn với Giant")

In [ ]:
# --- Thống kê theo folder --------------------------------------------
from collections import defaultdict
import json
per_folder = defaultdict(lambda: [0, 0])   # folder -> [số video, số frame]
for j in Path(FRAMEIDX_DIR).glob("*.json"):
    d = json.loads(j.read_text(encoding="utf-8"))
    per_folder[d["folder"]][0] += 1
    per_folder[d["folder"]][1] += d["n"]

print(f"{'Folder':<20} {'Video':>6} {'Keyframes':>11} {'Dự kiến':>10}")
print("-" * 52)
gv = gf = 0
for k in sorted(per_folder):
    v, n = per_folder[k]
    gv += v; gf += n
    print(f"{k:<20} {v:>6} {n:>11,} {FOLDER_IMAGE_COUNTS.get(k, 0):>10,}")
print("-" * 52)
print(f"{'TỔNG':<20} {gv:>6} {gf:>11,}")

## 8. Đóng gói kết quả

Tải `siglip2-giant-opt-patch16-384_batch{N}.zip` về, giải nén vào thư mục embedding riêng.
Mỗi file `.npy` có dạng `[N_keyframes, 1536]` float16, L2-normalized. KHÔNG merge
vào `Feature_Dataset/clip-features-32-aic25-b1/clip-features-32/` vì thư mục đó là CLIP 512 chiều.
Hoặc chọn "Save Version" rồi tạo Kaggle Dataset từ output để dùng với retrieval index 1536 chiều.

In [ ]:
import shutil, os
tag = f"batch{BATCH_ID}" if not FOLDERS_OVERRIDE else "custom"
if ZIP_RESULT:
    z1 = shutil.make_archive(f"/kaggle/working/{_tag}_{tag}", "zip", root_dir=OUTPUT_DIR)
    z2 = shutil.make_archive(f"/kaggle/working/{_tag}-frameidx_{tag}", "zip", root_dir=FRAMEIDX_DIR)
    for z in (z1, z2):
        print(f"{z}  ({os.path.getsize(z) / 1024**2:.1f} MB)")

for f in SHARD_FILES:      # dọn file tạm để output gọn
    try:
        os.remove(f)
    except OSError:
        pass
print("\nHoàn tất đợt", BATCH_ID, "- đổi BATCH_ID và chạy lại cho các đợt còn lại.")